# 04. CNN Training

This notebook trains the baseline convolutional neural network (CNN) developed for binary classification of cracked and uncracked concrete images from the SDNET2018 dataset.

The notebook covers model initialization, training, validation, model checkpointing and recording of training history. Model evaluation on the independent test set is performed in the subsequent evaluation notebook.

## 1. Imports and Reproducibility

In [1]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.13.0+cpu
Device: cpu


## 2. Load Dataset Metadata

In [2]:
from pathlib import Path

METADATA_DIR = Path("../data/metadata")

train_df = pd.read_csv(METADATA_DIR / "train.csv")
val_df = pd.read_csv(METADATA_DIR / "validation.csv")
test_df = pd.read_csv(METADATA_DIR / "test.csv")

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

Training images: 39264
Validation images: 8414
Test images: 8414


## 3. Define Image Transformations

In [3]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(
        10,
        interpolation=InterpolationMode.BILINEAR
    ),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 4. Create PyTorch Datasets

In [6]:
from torch.utils.data import Dataset
from PIL import Image

class ConcreteCrackDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(row["image_path"]).convert("RGB")
        label = torch.tensor(row["class_id"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
train_dataset = ConcreteCrackDataset(train_df, train_transform)
val_dataset = ConcreteCrackDataset(val_df, eval_transform)
test_dataset = ConcreteCrackDataset(test_df, eval_transform)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Training dataset: 39264
Validation dataset: 8414
Test dataset: 8414


## 5. Create DataLoaders

In [8]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 1227
Validation batches: 263
Test batches: 263


## 6. Define the Baseline CNN

In [9]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## 7. Initialize the Model

In [10]:
model = BaselineCNN().to(device)

print(model)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): AdaptiveAvgPool2d(output_size=(1, 1))
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)


## 8. Define Loss Function and Optimizer

In [11]:
num_uncracked = (train_df["class_id"] == 0).sum()
num_cracked = (train_df["class_id"] == 1).sum()

pos_weight = num_uncracked / num_cracked

pos_weight_tensor = torch.tensor(
    [pos_weight],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Positive class weight:", round(pos_weight, 4))
print("Learning rate:", optimizer.param_groups[0]["lr"])

Positive class weight: 5.6112
Learning rate: 0.001


## 9. Define Training and Validation Functions

In [12]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [13]:
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

## 10. Set Training Parameters

In [14]:
NUM_EPOCHS = 10

print("Epochs:", NUM_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", optimizer.param_groups[0]["lr"])

Epochs: 10
Batch size: 32
Learning rate: 0.001


## 11. Train the Baseline CNN

In [16]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_accuracy = validate(
        model,
        val_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "../trained_models/baseline_cnn_best.pth"
        )

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

KeyboardInterrupt: 

## 12. Training Speed Check

In [17]:
import time

model.train()

start_time = time.time()

images, labels = next(iter(train_loader))

elapsed = time.time() - start_time

print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)
print("Time to load one batch:", round(elapsed, 2), "seconds")

Batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Time to load one batch: 0.4 seconds


## 13. Training Speed Check

In [18]:
import time

model.train()

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device).unsqueeze(1)

start_time = time.time()

optimizer.zero_grad()

outputs = model(images)
loss = criterion(outputs, labels)

loss.backward()
optimizer.step()

elapsed = time.time() - start_time

print("Batch shape:", images.shape)
print("Training loss:", round(loss.item(), 4))
print("Time for one training batch:", round(elapsed, 2), "seconds")

Batch shape: torch.Size([32, 3, 224, 224])
Training loss: 1.2034
Time for one training batch: 1.76 seconds
